# COMPANY_DIM — Type 2 SCD Load (L1 → L3)

**Lakehouse target.** SparkSQL only. No monolithic `MERGE` — staged temp views,
one `UPDATE` to expire changed rows, one `INSERT` for new + new versions.

| | |
|---|---|
| **Source** | `<p_source_database_name>.<p_source_schema_name>.l1_company_dly` |
| **Target** | `<p_target_database_name>.<p_target_schema_name>.l3_DM_COR_COMPANY_DIM` |
| **Natural key** | `Company_ID` |
| **Tracked attributes (hash)** | `Company_WID, Company_Name, Company_Code, Business_Unit, Company_Subtype, Company_Currency` |
| **Source system** | `WORKDAY` |
| **Author** | MIA · for Chris Braden · branch `claude/friendly-cannon` |

> The `COMPANY_SK` primary key is an IDENTITY column and is autopopulated — the
> `INSERT` deliberately omits it.


## Phase 0 — Notebook parameters

These are exposed as Fabric pipeline parameters. The cell below declares them
as Spark session variables so every downstream cell is self-contained. Replace
the `SET VAR` literals with your pipeline values when running interactively.


In [ ]:
DECLARE OR REPLACE VARIABLE p_run_date             DATE;
DECLARE OR REPLACE VARIABLE p_run_ts               TIMESTAMP;
DECLARE OR REPLACE VARIABLE p_etl_run_id           BIGINT;
DECLARE OR REPLACE VARIABLE p_source_database_name STRING;
DECLARE OR REPLACE VARIABLE p_source_schema_name   STRING;
DECLARE OR REPLACE VARIABLE p_target_database_name STRING;
DECLARE OR REPLACE VARIABLE p_target_schema_name   STRING;

SET VAR p_run_date             = DATE'2026-04-24';
SET VAR p_run_ts               = TIMESTAMP'2026-04-24 06:00:00';
SET VAR p_etl_run_id           = 20260424060000;
SET VAR p_source_database_name = 'lh_hr_l1';
SET VAR p_source_schema_name   = 'workday';
SET VAR p_target_database_name = 'lh_hr_l3';
SET VAR p_target_schema_name   = 'dm_cor';

## Phase 1 — Stage the source feed with computed SCD hash

Reads `l1_company_dly`, `TRIM`s every string, `COALESCE`s NULLs to the sentinel
`Ø` and computes `SCD_HASH_CD` (MD5) over the **tracked attributes only** —
audit columns are deliberately excluded from the hash.


In [ ]:
CREATE OR REPLACE TEMPORARY VIEW v_stg_company AS
SELECT
    -- natural key + business attributes (trimmed)
    TRIM(src.Company_ID)         AS COMPANY_ID,
    TRIM(src.Company_Code)       AS COMPANY_CD,
    TRIM(src.Company_WID)        AS COMPANY_WID,
    TRIM(src.Company_Name)       AS COMPANY_NM_DESCR,
    TRIM(src.Company_Subtype)    AS COMPANY_SUBTYPE_CD,
    TRIM(src.Company_Currency)   AS COMPANY_CURRENCY_CD,
    TRIM(src.Business_Unit)      AS BUSINESS_UNIT_HIERARCHY_CD,
    -- columns with no apparent source field — insert NULL per spec
    CAST(NULL AS STRING)         AS COMPANY_HIERARCHY_DESCR,
    -- SCD hash over tracked attributes only (audit fields excluded)
    MD5(CONCAT_WS('||',
        COALESCE(TRIM(src.Company_WID),      'Ø'),
        COALESCE(TRIM(src.Company_Name),     'Ø'),
        COALESCE(TRIM(src.Company_Code),     'Ø'),
        COALESCE(TRIM(src.Business_Unit),    'Ø'),
        COALESCE(TRIM(src.Company_Subtype),  'Ø'),
        COALESCE(TRIM(src.Company_Currency), 'Ø')
    ))                           AS SCD_HASH_CD
FROM IDENTIFIER(p_source_database_name || '.' || p_source_schema_name || '.l1_company_dly') AS src
WHERE src.Company_ID IS NOT NULL;

## Phase 2 — Snapshot the current target rows

Captures the current open version (`CURRENT_RECORD_IND = 'Y'`) of every
`COMPANY_ID`. Used by Phase 3 to classify rows.


In [ ]:
CREATE OR REPLACE TEMPORARY VIEW v_tgt_current AS
SELECT
    COMPANY_ID,
    SCD_HASH_CD AS TGT_SCD_HASH_CD
FROM IDENTIFIER(p_target_database_name || '.' || p_target_schema_name || '.l3_DM_COR_COMPANY_DIM')
WHERE CURRENT_RECORD_IND = 'Y';

## Phase 3 — Classify staged rows: NEW / CHANGED / NO-OP

Single `LEFT JOIN` of staging against current target. Three disjoint sets:

* **NEW**     — `Company_ID` not present in current target
* **CHANGED** — present but hash differs
* **NO-OP**   — present and hash matches (filtered out)


In [ ]:
CREATE OR REPLACE TEMPORARY VIEW v_stg_classified AS
SELECT
    s.*,
    CASE
        WHEN t.COMPANY_ID IS NULL                 THEN 'NEW'
        WHEN t.TGT_SCD_HASH_CD <> s.SCD_HASH_CD   THEN 'CHANGED'
        ELSE 'NO-OP'
    END AS scd_action
FROM v_stg_company s
LEFT JOIN v_tgt_current t
       ON t.COMPANY_ID = s.COMPANY_ID;

CREATE OR REPLACE TEMPORARY VIEW v_stg_to_apply AS
SELECT * FROM v_stg_classified WHERE scd_action IN ('NEW','CHANGED');

CREATE OR REPLACE TEMPORARY VIEW v_stg_changed AS
SELECT COMPANY_ID FROM v_stg_classified WHERE scd_action = 'CHANGED';

## Phase 4 — Expire CHANGED rows in the target

For every CHANGED `Company_ID`, close out the current open version:

* `VALID_TO_TS        = p_run_date`
* `CURRENT_RECORD_IND = 'N'`
* `ETL_UPDATE_TS      = p_run_ts`
* `ETL_UPDATE_NUM     = p_etl_run_id`

This is a Delta `UPDATE` — not a `MERGE` — so the staged-approach rule holds.


In [ ]:
UPDATE IDENTIFIER(p_target_database_name || '.' || p_target_schema_name || '.l3_DM_COR_COMPANY_DIM') AS tgt
   SET VALID_TO_TS        = CAST(p_run_date AS TIMESTAMP),
       CURRENT_RECORD_IND = 'N',
       ETL_UPDATE_TS      = p_run_ts,
       ETL_UPDATE_NUM     = CAST(p_etl_run_id AS INT)
 WHERE tgt.CURRENT_RECORD_IND = 'Y'
   AND tgt.COMPANY_ID IN (SELECT COMPANY_ID FROM v_stg_changed);

## Phase 5 — Insert NEW rows + new versions of CHANGED rows

`COMPANY_SK` is an IDENTITY column and is **deliberately omitted** from the
column list — Fabric autopopulates it. Audit columns are stamped from
parameters. `VALID_FROM_TS` is clamped to `MAX(p_run_date, 1800-01-01)` per
the earliest-baseline rule.


In [ ]:
INSERT INTO IDENTIFIER(p_target_database_name || '.' || p_target_schema_name || '.l3_DM_COR_COMPANY_DIM') (
    COMPANY_ID,
    COMPANY_CD,
    COMPANY_WID,
    VALID_FROM_TS,
    VALID_TO_TS,
    CURRENT_RECORD_IND,
    SOFT_DELETE_FLG,
    SOURCE_SYSTEM_CD,
    SOURCE_EXTRACT_TS,
    ETL_CREATE_TS,
    ETL_UPDATE_TS,
    ETL_CREATE_NUM,
    ETL_UPDATE_NUM,
    SCD_HASH_CD,
    COMPANY_NM_DESCR,
    COMPANY_SUBTYPE_CD,
    COMPANY_CURRENCY_CD,
    COMPANY_HIERARCHY_DESCR,
    BUSINESS_UNIT_HIERARCHY_CD
)
SELECT
    s.COMPANY_ID,
    s.COMPANY_CD,
    s.COMPANY_WID,
    CAST(GREATEST(p_run_date, DATE'1800-01-01') AS TIMESTAMP) AS VALID_FROM_TS,
    TIMESTAMP'9999-12-31 00:00:00'                            AS VALID_TO_TS,
    'Y'                                                       AS CURRENT_RECORD_IND,
    'N'                                                       AS SOFT_DELETE_FLG,
    'WORKDAY'                                                 AS SOURCE_SYSTEM_CD,
    p_run_ts                                                  AS SOURCE_EXTRACT_TS,
    p_run_ts                                                  AS ETL_CREATE_TS,
    p_run_ts                                                  AS ETL_UPDATE_TS,
    CAST(p_etl_run_id AS INT)                                 AS ETL_CREATE_NUM,
    CAST(p_etl_run_id AS INT)                                 AS ETL_UPDATE_NUM,
    s.SCD_HASH_CD,
    s.COMPANY_NM_DESCR,
    s.COMPANY_SUBTYPE_CD,
    s.COMPANY_CURRENCY_CD,
    s.COMPANY_HIERARCHY_DESCR,
    s.BUSINESS_UNIT_HIERARCHY_CD
FROM v_stg_to_apply s;

## Phase 6 — Run summary (counts by action)

Diagnostic readout so the operator can confirm the run did what they expect.
Safe in production — it only reads from the temp views.


In [ ]:
SELECT
    scd_action,
    COUNT(*) AS row_count
FROM v_stg_classified
GROUP BY scd_action
ORDER BY scd_action;

## Phase 7 — Post-load integrity checks

Both checks should return **zero rows** in a healthy run:

1. Any `Company_ID` with more than one open version.
2. Any row where `VALID_FROM_TS > VALID_TO_TS`.


In [ ]:
-- Check 1: COMPANY_IDs with more than one open row (should be 0)
SELECT COMPANY_ID, COUNT(*) AS open_rows
FROM   IDENTIFIER(p_target_database_name || '.' || p_target_schema_name || '.l3_DM_COR_COMPANY_DIM')
WHERE  CURRENT_RECORD_IND = 'Y'
GROUP  BY COMPANY_ID
HAVING COUNT(*) > 1;

In [ ]:
-- Check 2: rows where VALID_FROM_TS > VALID_TO_TS (should be 0)
SELECT COUNT(*) AS bad_validity_rows
FROM   IDENTIFIER(p_target_database_name || '.' || p_target_schema_name || '.l3_DM_COR_COMPANY_DIM')
WHERE  VALID_FROM_TS > VALID_TO_TS;